# Goal 2 — Implementação e carregamento em MongoDB

Neste notebook criamos as 6 coleções do MongoDB (`movies`, `genres`, `movie_genres`, `users`, `ratings` e `tags`), iguais às tabelas do MySQL, a partir dos CSVs limpos em `data/processed/`. Medimos o tempo de carregamento e verificamos se os dados ficaram completos.

In [9]:
#%pip install pymongo

Importar as bibliotecas e ligar ao MongoDB.

In [10]:
import pandas as pd
import time
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["movielens"]
print(db.list_collection_names())

['ratings', 'movies', 'genres', 'tags', 'users', 'movie_genres']


## 1. Ler os dados

Lemos os 6 CSVs já limpos de `data/processed/` e transformamos cada linha num documento. Os valores em falta passam a `None` (no MongoDB ficam `null`) e as datas são lidas como datas.

In [11]:
ficheiros = [
    ("movies", {"movieId": "Int64", "title": "str", "year": "Int64", "imdbId": "str", "tmdbId": "Int64"}, None),
    ("genres", {"genre": "str", "genreId": "Int64"}, None),
    ("movie_genres", {"movieId": "Int64", "genreId": "Int64"}, None),
    ("users", {"userId": "Int64", "latitude": "Float64", "longitude": "Float64"}, None),
    ("ratings", {"userId": "Int64", "movieId": "Int64", "rating": "Float64"}, ["timestamp"]),
    ("tags", {"userId": "Int64", "movieId": "Int64", "tag": "str"}, ["timestamp"])
]
dados = {}

for nome, tipos, data in ficheiros:
    df = pd.read_csv(f"../data/processed/{nome}.csv", dtype=tipos, parse_dates = data)
    dados[nome] = df.astype(object).where(df.notna(), None).to_dict("records")
    print(nome, len(dados[nome]))

dados["users"] = list(map(lambda user: {
    "userId": user["userId"],
    "location": {
        "type": "Point",
        "coordinates": [ user["latitude"], user["longitude" ] ]
    }
}, dados["users"]))
print(dados["ratings"][0])

movies 9742
genres 19
movie_genres 22050
users 610
ratings 100000
tags 3683
{'userId': 432, 'movieId': 77866, 'rating': 4.5, 'timestamp': Timestamp('2012-04-23 00:07:21')}


In [12]:
print(dados["users"][0])

{'userId': 1, 'location': {'type': 'Point', 'coordinates': [40.71862834068687, -7.827198594667862]}}


## 2. Criar as coleções

Criamos as 6 coleções com regras de validação: os campos obrigatórios e o tipo de cada campo (por exemplo, o `rating` tem de estar entre 0.5 e 5). É o equivalente aos tipos e ao `NOT NULL` do `CREATE TABLE` no MySQL.

O MongoDB não tem chaves primárias como o MySQL; cada documento só tem o `_id`, gerado automaticamente. Para ter as mesmas regras do MySQL, criamos um índice único nos mesmos campos das chaves primárias (por exemplo, `userId` + `movieId` nos `ratings`). Assim, o MongoDB também recusa documentos repetidos, e as duas bases de dados ficam equivalentes desde o início, incluindo na comparação dos tempos de carregamento.

In [13]:
# Remover as coleções antigas
for nome in ["movies", "genres", "movie_genres", "users", "ratings", "tags"]:
    if nome in db.list_collection_names():
        db[nome].drop()

print("Coleções antigas removidas com sucesso!")


Coleções antigas removidas com sucesso!


In [14]:
regras = {
    "movies": {
        "required": ["movieId", "title"],
        "properties": {
            "movieId": {"bsonType": "int"},
            "title":   {"bsonType": "string"},
            "year":    {"bsonType": ["int", "null"]},
            "imdbId":  {"bsonType": "string"},
            "tmdbId":  {"bsonType": ["int", "null"]}
        }
    },
    "genres": {
        "required": ["genreId", "genre"],
        "properties": {
            "genreId": {"bsonType": "int"},
            "genre":   {"bsonType": "string"}
        }
    },
    "movie_genres": {
        "required": ["movieId", "genreId"],
        "properties": {
            "movieId": {"bsonType": "int"},
            "genreId": {"bsonType": "int"}
        }
    },
    "users": {
        "required": ["userId", "location"],
        "properties": {
            "userId":  {"bsonType": "int"},
            "location": {"bsonType": "object"}
        }
    },
    "ratings": {
        "required": ["userId", "movieId", "rating", "timestamp"],
        "properties": {
            "userId":    {"bsonType": "int"},
            "movieId":   {"bsonType": "int"},
            "rating":    {"bsonType": "double", "minimum": 0.5, "maximum": 5},
            "timestamp": {"bsonType": "date"}
        }
    },
    "tags": {
        "required": ["userId", "movieId", "tag", "timestamp"],
        "properties": {
            "userId":    {"bsonType": "int"},
            "movieId":   {"bsonType": "int"},
            "tag":       {"bsonType": "string"},
            "timestamp": {"bsonType": "date"}
        }
    }
}

chaves = {
    "movies":       ["movieId"],
    "genres":       ["genreId"],
    "users":        ["userId"],
    "movie_genres": ["movieId", "genreId"],
    "ratings":      ["userId", "movieId"],
    "tags":         ["userId", "movieId", "tag"]
}

for nome in regras:
    db[nome].drop()
    db.create_collection(nome, validator={"$jsonSchema": {"bsonType": "object", **regras[nome]}})
    db[nome].create_index([(campo, 1) for campo in chaves[nome]], unique=True)

print(db.list_collection_names())

['ratings', 'movies', 'users', 'tags', 'genres', 'movie_genres']


## 3. Carregar os dados e medir o tempo

Inserimos os documentos em cada coleção com `insert_many` e medimos o tempo de cada uma e o total.

In [15]:
tempos_mongo = {}

for nome, docs in dados.items():
    inicio = time.time()
    db[nome].insert_many(docs)
    tempos_mongo[nome] = time.time() - inicio
    print(f"{nome}: {len(docs)} documentos em {tempos_mongo[nome]:.2f} s")

print(f"Total MongoDB: {sum(tempos_mongo.values()):.2f} s")

movies: 9742 documentos em 0.24 s
genres: 19 documentos em 0.00 s
movie_genres: 22050 documentos em 0.44 s
users: 610 documentos em 0.01 s
ratings: 100000 documentos em 2.61 s
tags: 3683 documentos em 0.08 s
Total MongoDB: 3.39 s


## 4. Verificar os dados

Confirmamos que cada coleção tem o mesmo número de documentos que o respetivo CSV e mostramos os índices que existem em cada uma.

In [16]:
for nome, docs in dados.items():
    total = db[nome].count_documents({})
    print(f"{nome}: CSV = {len(docs)}, MongoDB = {total}")
    
    guardado = db[nome].find_one({"_id": docs[0]["_id"]})
    assert guardado == docs[0], f"Dados diferentes em {nome}"

print()
for nome in dados:
    print(f"Índices de {nome}: {list(db[nome].index_information())}")

movies: CSV = 9742, MongoDB = 9742
genres: CSV = 19, MongoDB = 19
movie_genres: CSV = 22050, MongoDB = 22050
users: CSV = 610, MongoDB = 610
ratings: CSV = 100000, MongoDB = 100000
tags: CSV = 3683, MongoDB = 3683

Índices de movies: ['_id_', 'movieId_1']
Índices de genres: ['_id_', 'genreId_1']
Índices de movie_genres: ['_id_', 'movieId_1_genreId_1']
Índices de users: ['_id_', 'userId_1']
Índices de ratings: ['_id_', 'userId_1_movieId_1']
Índices de tags: ['_id_', 'userId_1_movieId_1_tag_1']
